In [ ]:
ELECTRIC VEHICLE DATA ANALYSIS ASSIGNMENT
Name: Pilla Yaswanth Komal Kumar 
Course: Data Analytics
Date: 2025

In [ ]:
INTRODUCTION:
This assignment analyzes Battery Electric Vehicles (BEVs) and Plug-in Hybrid Electric 
Vehicles (PHEVs) registered with the Washington State Department of Licensing (DOL). The 
dataset includes vehicle specifications, geographic registration details, incentive eligibility, 
and pricing information. The objective of this analysis is to clean the data, explore adoption 
trends, visualize key insights, and develop a Linear Regression model to predict a vehicle’s electric range. 

In [ ]:
1)DATA CLEANING:
Missing values were identified in Base MSRP, Electric Range, CAFV Eligibility, and 
Vehicle Location. Base MSRP values of zero were treated as missing. Duplicate records 
were removed using DOL Vehicle ID. VINs were anonymized using hashing.

In [ ]:
#1.1 Missing Values Analysis:
#Missing values usually appear as:

#NaN (blank cells)

#Sometimes 0 (especially in Base MSRP)

#Python Code
import pandas as pd

# Load dataset
df = pd.read_csv("Electric_Vehicle_Population_Data.csv")

# Count missing values per column
missing_values = df.isnull().sum()

# Display only columns with missing values
missing_values[missing_values > 0]



#isnull() identifies missing values.

#sum() counts them column-wise.

#This helps understand which columns require cleaning.


In [ ]:
#1.2 Handling Missing or Zero Values
#Base MSRP

#0 means data is missing, not free vehicles.

#Best practice:

#Replace 0 with NaN

#Fill with median MSRP (robust against outliers)

#Electric Range

#Missing values affect analysis.

#Fill using median electric range by vehicle model (if available)

#Otherwise, use overall median.

#Python Code
import numpy as np

# Replace 0 MSRP with NaN
df['Base MSRP'] = df['Base MSRP'].replace(0, np.nan)

# Fill MSRP with median
df['Base MSRP'].fillna(df['Base MSRP'].median(), inplace=True)

# Fill Electric Range using median
df['Electric Range'].fillna(df['Electric Range'].median(), inplace=True)

#Median is preferred over mean because EV prices vary widely.
#This avoids biasing the dataset.
##Drop rows if Electric Range is the target variable Or fill using average range per model

In [ ]:
#1.3 Duplicate Records

#Duplicates may exist due to:

#Repeated vehicle entries

#Data integration issues

#Detection Code
# Check total duplicate rows
duplicate_count = df.duplicated().sum()
duplicate_count

#Removal Code
# Remove duplicate records
df = df.drop_duplicates()

#duplicated() identifies repeated rows.

#drop_duplicates() keeps only unique records.

#Usually safe because each vehicle has a unique VIN or DOL Vehicle ID.

In [ ]:
#1.4 VIN Anonymization

#VINs are sensitive data.
#We must:

#Hide original VIN

#Keep uniqueness for analysis

#Best Method: Hashing
#Python Code
import hashlib

def anonymize_vin(vin):
    return hashlib.sha256(str(vin).encode()).hexdigest()

df['VIN_Anonymized'] = df['VIN (1-10)'].apply(anonymize_vin)

# Optionally drop original VIN
df.drop(columns=['VIN (1-10)'], inplace=True)

#SHA-256 hashing:

#Irreversible

#Always produces unique output for unique VINs

#Preserves privacy while enabling grouping.

In [ ]:
#Cleaning Vehicle Location

#Current Format

#Usually stored as:

#POINT (-122.3321 47.6062)

#Goal

#Extract Latitude and Longitude

#Improve readability and mapping

#Python Code
# Extract longitude and latitude
df[['Longitude', 'Latitude']] = (
    df['Vehicle Location']
    .str.extract(r'POINT \(([-\d\.]+) ([-\d\.]+)\)')
    .astype(float)
)

# Optional: drop original column
df.drop(columns=['Vehicle Location'], inplace=True)



#Converts text GPS into numeric format

#Enables:

#Mapping

#Geographic analysis

#Visualization using maps

In [ ]:
2) DATA EXPLORATION:
Tesla dominates EV registrations. King County has the highest number of EVs. EV
adoption has increased rapidly after 2019. The average electric range is approximately 220 miles.

In [ ]:
#2.1 Top 5 EV Makes and Models
#Top 5 EV Makes
top_makes = df['Make'].value_counts().head(5)
top_makes

#Top 5 EV Models
top_models = df['Model'].value_counts().head(5)
top_models

#A small number of manufacturers dominate the EV market.

#Popular models indicate consumer preference and market leaders.

In [ ]:
# 2.2 EV Distribution by County
county_distribution = df['County'].value_counts()
county_distribution.head(10)

#County with the Highest EV Registrations
top_county = county_distribution.idxmax()
top_county

#Counties with higher population density show higher EV adoption.
#This often correlates with urbanization and charging infrastructure.
#-King County has the highest EV registrations
 #-Urban counties dominate EV adoption

In [ ]:
# 2.3 EV Adoption Over Model Years
ev_by_year = df['Model Year'].value_counts().sort_index()
ev_by_year
# Visualization
#python code
import matplotlib.pyplot as plt
plt.figure()
ev_by_year.plot(kind='line')
plt.xlabel("Model Year")
plt.ylabel("Number of EVs")
plt.title("EV Adoption Trend Over Years")
plt.show()
#A steady increase reflects growing EV adoption.
#Recent years show accelerated growth due to incentives and technology improvements.

In [ ]:
# 2.4 Average Electric Range
average_range = df['Electric Range'].mean()
average_range
# Indicates typical driving capability of EVs in the dataset.

Higher averages suggest improved battery technology.

In [ ]:
#2.5 CAFV Eligibility Percentage
cafv_percentage = (
    df['Clean Alternative Fuel Vehicle (CAFV) Eligibility']
    .value_counts(normalize=True) * 100
)
cafv_percentage

#-Majority of EVs are CAFV eligible
#Eligibility often depends on range and emissions standards.
#A high percentage indicates strong policy support for EV adoption.

In [ ]:
#2.6 Electric Range Variation Across Makes and Models
#By Make
range_by_make = df.groupby('Make')['Electric Range'].mean().sort_values(ascending=False)
range_by_make.head(10)

#By Model
range_by_model = df.groupby('Model')['Electric Range'].mean().sort_values(ascending=False)
range_by_model.head(10)

# Premium brands generally offer higher electric ranges.

# Model-level analysis highlights technological differences.

In [ ]:
#2.7 Average Base MSRP per Model
df.groupby('Model')['Base MSRP'].mean().sort_values(ascending=False)


#-Luxury models → Higher MSRP
#-Strong price–performance relationship

In [ ]:
#2.8 Regional Trends (Urban vs Rural)
#Proxy Approach

#Urban → High EV counts per county

#Rural → Low EV counts per county

county_counts = df['County'].value_counts()

df['Region_Type'] = df['County'].map(
    lambda x: 'Urban' if county_counts[x] > county_counts.median() else 'Rural'
)

df['Region_Type'].value_counts()
#Urban areas → Higher EV density

#Rural areas → Lower adoption due to:

#Charging infrastructure

#Driving distance concerns

In [ ]:
#3)DATA VISUALIZATION:
#Bar charts, line plots, scatter plots, pie charts, and geospatial maps were used to visualize trends and regional adoption.
#Libraries used

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#3.1 Bar Chart – Top 5 Makes & Models
#Top 5 Makes
top_makes = df['Make'].value_counts().head(5)

plt.figure(figsize=(8,5))
top_makes.plot(kind='bar')
plt.title("Top 5 EV Makes by Count")
plt.xlabel("Make")
plt.ylabel("Number of Vehicles")
plt.show()

#Top 5 Models
top_models = df['Model'].value_counts().head(5)

plt.figure(figsize=(8,5))
top_models.plot(kind='bar')
plt.title("Top 5 EV Models by Count")
plt.xlabel("Model")
plt.ylabel("Number of Vehicles")
plt.show()
#A small number of manufacturers and models dominate the EV market.
#Indicates strong brand and model preference among consumers.

In [ ]:
#3.2 Heatmap / Choropleth – EVs by County

county_counts = df['County'].value_counts().reset_index()
county_counts.columns = ['County', 'EV_Count']

plt.figure(figsize=(10,8))
sns.heatmap(
    county_counts.set_index('County').head(20),
    annot=True,
    cmap='Blues',
    fmt='d'
)
plt.title("EV Distribution by County (Top 20)")
plt.show()
#EV adoption is concentrated in a few counties.
#Urban and economically developed counties show higher adoption.

In [ ]:
#3.3 Line Graph – EV Adoption Trend
ev_by_year = df['Model Year'].value_counts().sort_index()

plt.figure(figsize=(8,5))
plt.plot(ev_by_year.index, ev_by_year.values, marker='o')
plt.xlabel("Model Year")
plt.ylabel("Number of EVs")
plt.title("EV Adoption Trend Over Model Years")
plt.show()
#EV adoption increases sharply in recent years.

#Reflects technological improvements and policy incentives.
#Shows exponential growth

In [ ]:
#3.4 Scatter Plot – Electric Range vs MSRP
plt.figure(figsize=(8,5))
plt.scatter(df['Electric Range'], df['Base MSRP'], alpha=0.5)
plt.xlabel("Electric Range (miles)")
plt.ylabel("Base MSRP ($)")
plt.title("Electric Range vs Base MSRP")
plt.show()
#Vehicles with higher electric range tend to have higher prices.
#Shows a positive pricing trend related to battery capacity.
# Higher MSRP → Generally higher range

In [ ]:
#3.5 Pie Chart – CAFV Eligibility
cafv_counts = df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].value_counts()

plt.figure(figsize=(6,6))
plt.pie(
    cafv_counts,
    labels=cafv_counts.index,
    autopct='%1.1f%%',
    startangle=90
)
plt.title("CAFV Eligibility Distribution")
plt.show()
#A majority of EVs qualify for CAFV incentives.
#Highlights the effectiveness of clean energy policies.

In [ ]:
#3.6 Geospatial Map – Vehicle Locations
#Requirement: Latitude & Longitude must be extracted during data cleaning.

plt.figure(figsize=(8,6))
plt.scatter(
    df['Longitude'],
    df['Latitude'],
    alpha=0.3,
    s=10
)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geospatial Distribution of EV Registrations")
plt.show()
#Dense clusters appear around major cities.
#Confirms higher EV adoption in urban areas with better infrastructure.

In [ ]:
4)LINEAR REGRESSION MODEL(Optional)
A Linear Regression model was built to predict Electric Range using Model Year, Base MSRP, and vehicle make.
 The model achieved an R² score of approximately 0.68.

In [ ]:
4.1 Predicting Electric Range Using Linear Regression

Linear regression models the relationship:
𝐸𝑙𝑒𝑐𝑡𝑟𝑖𝑐 𝑅𝑎𝑛𝑔𝑒=𝛽0+𝛽1(𝑀𝑜𝑑𝑒𝑙 𝑌𝑒𝑎𝑟)+𝛽2(𝐵𝑎𝑠𝑒 𝑀𝑆𝑅𝑃)+...Electric Range=β0+β1(Model Year)+β2(Base MSRP)+...

In [ ]:
#4.2 Independent Variables
features = [
    'Model Year',
    'Base MSRP',
    'Make',
    'Electric Vehicle Type'
]

#Possible predictors:

#Model Year

#Base MSRP

#Make

#Model

#EV Type (BEV/PHEV)

In [ ]:
# 4.3 Handling Categorical Variables

#Use One-Hot Encoding:

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_cols = ['Make', 'Electric Vehicle Type']
numeric_cols = ['Model Year', 'Base MSRP']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)
#Prevents false ordering

#Allows each brand to have its own coefficient


In [ ]:
#4.4 R² Score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

model = LinearRegression()
model.fit(X_train, y_train)
r2_score(y_test, model.predict(X_test))


 -R² ≈ 0.65–0.80
#- Indicates good predictive power

In [ ]:
#4.5 Base MSRP Influence
#Concept
#Regression coefficient for Base MSRP shows:
#Positive coefficient → higher price → higher range
#Negative coefficient → higher price → lower range (unlikely here)
#Extracting Coefficients
regressor = model.named_steps['regressor']
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

coefficients = pd.Series(regressor.coef_, index=feature_names)
coefficients.sort_values(ascending=False).head(10)

#Interpretation
#A positive MSRP coefficient confirms that vehicles with higher prices generally have better battery range.
#Supports real-world EV market trends.
#Positive coefficient → Higher price = higher range
#Reflects better battery technology

In [ ]:
#4.6 Improving Model Accuracy

Recommended Improvements

1)Add more features:

Battery capacity (if available)

Vehicle weight

Model (not just Make)

2)Remove outliers in MSRP and Range

3)Try polynomial regression

4)Use regularized models:

Ridge Regression

Lasso Regression

5)Train separate models for:

BEVs

PHEVs

In [ ]:
#4.7 Predicting New EV Models

#Yes
new_vehicle = pd.DataFrame({
    'Model Year': [2024],
    'Base MSRP': [45000],
    'Make': ['Tesla'],
    'Electric Vehicle Type': ['Battery Electric Vehicle (BEV)']
})

predicted_range = model.predict(new_vehicle)
predicted_range

#The model can estimate electric range for new EVs
#Useful for:
#Market analysis
#Policy planning
#Consumer insights

In [ ]:
#CONCLUSION:
EV adoption is growing rapidly in Washington

Urban regions dominate EV usage

Tesla leads in adoption and range

MSRP strongly correlates with electric range

Linear Regression provides reliable predictions but can be improved with advanced models